# 01 - Preprocessing ISOT

Output:
- `src/preprocessing.py`
- `data/processed/preprocessed_isot_full.csv`
- `data/preprocessed_isot_full.csv` for backward compatibility
- `reports/01_preprocessing/preprocessing_summary.json`

This notebook downloads/loads ISOT, fixes Reuters leakage, keeps negation words through
`src.preprocessing.preprocess_text`, and saves a reusable processed dataset.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

from src.preprocessing import preprocess_text

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORT_DIR = ROOT / "reports" / "01_preprocessing"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def find_isot_files():
    candidates = [
        DATA_RAW,
        DATA_RAW / "isot",
        DATA_RAW / "News_Dataset",
        DATA_RAW / "isot-fake-news-dataset",
    ]
    for base in candidates:
        fake_path = base / "Fake.csv"
        true_path = base / "True.csv"
        if fake_path.exists() and true_path.exists():
            return fake_path, true_path
    return None, None


fake_path, true_path = find_isot_files()

if fake_path is None or true_path is None:
    import kagglehub

    kaggle_path = Path(kagglehub.dataset_download("rahulogoel/isot-fake-news-dataset"))
    possible_pairs = [
        (kaggle_path / "News_Dataset" / "Fake.csv", kaggle_path / "News_Dataset" / "True.csv"),
        (kaggle_path / "Fake.csv", kaggle_path / "True.csv"),
    ]
    for fake_candidate, true_candidate in possible_pairs:
        if fake_candidate.exists() and true_candidate.exists():
            fake_path, true_path = fake_candidate, true_candidate
            break

if fake_path is None or true_path is None:
    raise FileNotFoundError(
        "Could not find ISOT Fake.csv/True.csv. Put them in data/raw/isot/ "
        "or allow kagglehub to download rahulogoel/isot-fake-news-dataset."
    )

print("Fake CSV:", fake_path)
print("True CSV:", true_path)

fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)

print("Fake shape:", fake_df.shape)
print("True shape:", true_df.shape)
display(fake_df.head(2))
display(true_df.head(2))


/Users/duyhung/Desktop/CHUNG/HUST/Fake-News-Detection/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Resuming download from 5242880 bytes (37733083 bytes left)...


Resuming download from https://www.kaggle.com/api/v1/datasets/download/rahulogoel/isot-fake-news-dataset?dataset_version_number=1 (5242880/42975963) bytes left.


 12%|█▏        | 5.00M/41.0M [00:00<?, ?B/s]

 15%|█▍        | 6.00M/41.0M [00:01<00:44, 818kB/s]

...

Extracting files...


Fake CSV: /Users/duyhung/.cache/kagglehub/datasets/rahulogoel/isot-fake-news-dataset/versions/1/News_Dataset/Fake.csv
True CSV: /Users/duyhung/.cache/kagglehub/datasets/rahulogoel/isot-fake-news-dataset/versions/1/News_Dataset/True.csv


Fake shape: (23481, 4)
True shape: (21417, 4)


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"


In [3]:
raw_summary = {
    "fake_shape": list(fake_df.shape),
    "true_shape": list(true_df.shape),
    "fake_nulls": fake_df.isnull().sum().to_dict(),
    "true_nulls": true_df.isnull().sum().to_dict(),
    "fake_duplicates": int(fake_df.duplicated().sum()),
    "true_duplicates": int(true_df.duplicated().sum()),
}

fake_df = fake_df.drop_duplicates().copy()
true_df = true_df.drop_duplicates().copy()

fake_df = fake_df[fake_df["text"].astype(str).str.strip() != ""].copy()
true_df = true_df[true_df["text"].astype(str).str.strip() != ""].copy()

true_df["label"] = 0  # REAL
fake_df["label"] = 1  # FAKE

df = pd.concat([true_df, fake_df], ignore_index=True)
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)
df["full_text"] = (df["title"].str.strip() + " " + df["text"].str.strip()).str.strip()

df = df[df["full_text"] != ""].copy()
duplicated_full_text = int(df.duplicated(subset=["full_text"]).sum())
conflicting_texts = int((df.groupby("full_text")["label"].nunique() > 1).sum())
df = df.drop_duplicates(subset=["full_text"], keep="first").reset_index(drop=True)

print("After basic cleaning:", df.shape)
print(df["label"].value_counts().sort_index())
print("Duplicated full_text removed:", duplicated_full_text)
print("Conflicting full_text count:", conflicting_texts)


After basic cleaning: (38656, 6)
label
0    21195
1    17461
Name: count, dtype: int64
Duplicated full_text removed: 5402
Conflicting full_text count: 0


In [4]:
sample = "WASHINGTON (Reuters) - Trump did not listen to the warnings, never neither!"
print("Original :", sample)
print("Processed:", preprocess_text(sample))

df["processed_text"] = df["full_text"].apply(preprocess_text)
empty_processed = int((df["processed_text"].astype(str).str.strip() == "").sum())
df = df[df["processed_text"].astype(str).str.strip() != ""].reset_index(drop=True)

final_cols = ["title", "text", "subject", "date", "full_text", "processed_text", "label"]
final_cols = [col for col in final_cols if col in df.columns]
df = df[final_cols]

print("Final shape:", df.shape)
print(df["label"].value_counts().sort_index())
display(df[["full_text", "processed_text", "label"]].sample(3, random_state=42))


Original : WASHINGTON (Reuters) - Trump did not listen to the warnings, never neither!


Processed: washington trump not listen warning never neither


Final shape: (38651, 7)
label
0    21195
1    17456
Name: count, dtype: int64


,full_text,processed_text,label
35902,BURIED BY MEDIA: Aide To Leftist US Congressma...,buried medium aide leftist congressman sander ...,1
29115,TN Lawmakers Got Big Money To Approve 279 Perc...,lawmaker got big money approve percent interes...,1
38280,Patrick Henningsen LIVE with guest Sean Stone ...,patrick henningsen live guest sean stone proje...,1


In [5]:
processed_path = DATA_PROCESSED / "preprocessed_isot_full.csv"
compat_path = ROOT / "data" / "preprocessed_isot_full.csv"
compact_path = DATA_PROCESSED / "preprocessed_isot.csv"

df.to_csv(processed_path, index=False)
df.to_csv(compat_path, index=False)
df[["processed_text", "label"]].to_csv(compact_path, index=False)

summary = {
    **raw_summary,
    "final_shape": list(df.shape),
    "final_label_distribution": df["label"].value_counts().sort_index().to_dict(),
    "empty_processed_removed": empty_processed,
    "duplicated_full_text_removed": duplicated_full_text,
    "conflicting_full_text_count": conflicting_texts,
    "processed_path": str(processed_path),
    "compat_path": str(compat_path),
}

with open(REPORT_DIR / "preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:", processed_path)
print("Saved:", compat_path)
print("Saved:", REPORT_DIR / "preprocessing_summary.json")


Saved: /Users/duyhung/Desktop/CHUNG/HUST/Fake-News-Detection/data/processed/preprocessed_isot_full.csv
Saved: /Users/duyhung/Desktop/CHUNG/HUST/Fake-News-Detection/data/preprocessed_isot_full.csv
Saved: /Users/duyhung/Desktop/CHUNG/HUST/Fake-News-Detection/reports/01_preprocessing/preprocessing_summary.json
